# 03 — SVD & PCA Analysis

Linear decomposition of the expression matrix:
- **SVD**: singular value decomposition to reveal latent structure
- **PCA**: principal component analysis to separate disease groups
- **UMAP**: non-linear embedding for visual exploration

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = '/content/gene-expression-la-analysis'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from linear_algebra import (
    run_svd, svd_explained_variance_table, select_n_components_by_variance,
    run_pca, pca_gene_contributions, biplot_loadings
)
from visualization import (
    plot_scree, plot_pca_scatter, plot_pca_3d,
    plot_gene_loadings, plot_biplot, plot_umap
)
from validation import pc_group_anova

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')


In [ ]:
# ── Load processed data ────────────────────────────────────────────────────
expr = pd.read_csv(f'{PROJECT_ROOT}/data/processed/processed_expression.csv',
                   index_col=0)
meta = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata_clean.csv',
                   index_col=0)

# Align samples
common = expr.columns.intersection(meta.index)
expr   = expr[common]
meta   = meta.loc[common]

print(f'Expression: {expr.shape} | Metadata: {meta.shape}')


In [ ]:
# ── Identify label column ─────────────────────────────────────────────────
# Inspect metadata columns to pick a disease/phenotype label
print(meta.dtypes)
print(meta.head(3))


In [ ]:
# !! ADJUST this to your actual metadata column !!
# For GSE2034, 'characteristics_ch1' often contains ER status etc.
LABEL_COL = 'source_name_ch1'   # <-- change as needed

if LABEL_COL in meta.columns:
    labels = meta[LABEL_COL]
else:
    # Fallback: create a dummy binary label
    labels = pd.Series(['GroupA'] * (len(common)//2) +
                        ['GroupB'] * (len(common) - len(common)//2),
                       index=common, name='group')

print('Label distribution:')
print(labels.value_counts())


## 1. SVD

In [ ]:
svd_result = run_svd(expr, n_components=50)
evr_table  = svd_explained_variance_table(svd_result)
print(evr_table.head(10).to_string(index=False))


In [ ]:
plot_scree(svd_result['explained_variance_ratio'],
           title='SVD — Explained Variance',
           save_path=f'{PROJECT_ROOT}/data/results/03_svd_scree.png')

n_sig = select_n_components_by_variance(svd_result, threshold=0.80)
print(f'Components for 80% variance: {n_sig}')


In [ ]:
# ── SVD singular values (log scale) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(svd_result['S'], 'o-', color='steelblue', ms=4)
ax.set_yscale('log')
ax.set_xlabel('Singular value rank')
ax.set_ylabel('Singular value (log scale)')
ax.set_title('SVD Singular Value Spectrum')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/03_svd_spectrum.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 2. PCA

In [ ]:
pca_result = run_pca(expr, n_components=50, center=True, scale=False)

plot_scree(pca_result['explained_variance_ratio'],
           title='PCA — Explained Variance',
           save_path=f'{PROJECT_ROOT}/data/results/03_pca_scree.png')


In [ ]:
# ── PC1 vs PC2 coloured by group ───────────────────────────────────────────
aligned_labels = labels.reindex(pca_result['scores'].index)

plot_pca_scatter(pca_result['scores'], labels=aligned_labels,
                 pc_x=1, pc_y=2,
                 title='PCA — PC1 vs PC2',
                 save_path=f'{PROJECT_ROOT}/data/results/03_pca_pc1_pc2.png')


In [ ]:
plot_pca_scatter(pca_result['scores'], labels=aligned_labels,
                 pc_x=1, pc_y=3,
                 title='PCA — PC1 vs PC3',
                 save_path=f'{PROJECT_ROOT}/data/results/03_pca_pc1_pc3.png')


In [ ]:
# ── 3D PCA ────────────────────────────────────────────────────────────────
plot_pca_3d(pca_result['scores'], labels=aligned_labels,
            title='PCA 3D — PC1/PC2/PC3')


In [ ]:
# ── Gene loadings for PC1 ─────────────────────────────────────────────────
plot_gene_loadings(pca_result['loadings']['PC1'], top_n=30,
                   title='PC1 Gene Loadings',
                   save_path=f'{PROJECT_ROOT}/data/results/03_pc1_loadings.png')


In [ ]:
# ── Biplot (PC1 vs PC2) ───────────────────────────────────────────────────
plot_biplot(pca_result['scores'], pca_result['loadings'],
            labels=aligned_labels, top_n_genes=15,
            save_path=f'{PROJECT_ROOT}/data/results/03_biplot.png')


In [ ]:
# ── ANOVA: which PCs associate with group? ────────────────────────────────
anova_df = pc_group_anova(pca_result['scores'].iloc[:, :20], aligned_labels)
print(anova_df[anova_df['padj_anova'] < 0.05].head(10))


## 3. UMAP

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    # Use PCA-reduced space for speed
    X_pca = pca_result['scores'].iloc[:, :30].values
    embedding = reducer.fit_transform(X_pca)

    plot_umap(embedding, labels=aligned_labels,
              title='UMAP of top-30 PCs',
              save_path=f'{PROJECT_ROOT}/data/results/03_umap.png')

    # Save embedding
    umap_df = pd.DataFrame(embedding, columns=['UMAP1','UMAP2'],
                           index=pca_result['scores'].index)
    umap_df.to_csv(f'{PROJECT_ROOT}/data/processed/umap_embedding.csv')
except ImportError:
    print('umap-learn not installed. Run: pip install umap-learn')


In [ ]:
# ── Save PCA results ──────────────────────────────────────────────────────
pca_result['scores'].to_csv(f'{PROJECT_ROOT}/data/processed/pca_scores.csv')
pca_result['loadings'].to_csv(f'{PROJECT_ROOT}/data/processed/pca_loadings.csv')
print('PCA results saved.')
